# Steam x Twitch - pipeline demo

The real work is in `pipeline.py` and the files in `src/`. This notebook shows the steps one by one
(for the presentation) and makes a first small analysis of the final dataset.

To only build the data, run `make` in the terminal (or `python pipeline.py`).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from path import *
from src.load import load_twitch, load_steam, load_steamcharts, load_list, load_dict
from src.clean import clean_twitch, clean_steam, remove_double_months, fill_steamcharts_gaps, flag_outliers
from src.match import clean_name, find_matches
from src.join import join_twitch_steam, split_matched, add_steamcharts
from src.transform import add_rolling_mean, add_features, add_groups, add_scaled, make_wide_table

## 1. Import (LE1)
The Twitch file has mixed encodings (utf-8 + cp1252), `load_twitch` handles this.

In [ ]:
twitch = load_twitch(Twitch_PATH)
steam = load_steam(Steam_PATH)
steamcharts = load_steamcharts(Steamcharts_FOLDER)

print("twitch:", twitch.shape, " steam:", steam.shape, " steamcharts:", steamcharts.shape)
twitch.head()

In [ ]:
# example: names with accents that were broken before the encoding fix
twitch[twitch["Game"].str.contains("é", na=False)]["Game"].unique()[:10]

## 2. Cleaning (LE2)

In [ ]:
not_games = load_list(Not_games_PATH, "twitch_name")
renamed = load_dict(Renamed_games_PATH, "twitch_name", "steam_name")

twitch = clean_twitch(twitch, not_games, renamed)
steam = clean_steam(steam)
steamcharts = fill_steamcharts_gaps(steamcharts)

In [ ]:
# example for the duplicate rule: Rainbow Six Siege is 5 times in the raw steam data, we keep 359550
raw_steam = load_steam(Steam_PATH)
raw_steam[raw_steam["name"].str.contains("Rainbow Six® Siege", na=False)][["appid", "name", "num_reviews_total"]]

## 3. Matching + join Twitch with Steam (LE4)

In [ ]:
print(clean_name("Tom Clancy's Rainbow Six® Siege"), "|", clean_name("Final Fantasy VII"))

matches = find_matches(twitch["clean_name"], steam["clean_name"])
df = join_twitch_steam(twitch, steam, matches)
df, not_in_steam = split_matched(df)
df = remove_double_months(df)

In [ ]:
# the fuzzy matches (score < 100) - we checked them by hand
fuzzy = df[df["steam_match_score"] < 100][["Game", "name", "steam_match_score"]].drop_duplicates()
fuzzy

In [ ]:
# the biggest twitch games that are not on steam
not_in_steam.head(10)

## 4. Join with SteamCharts + 5. Transform (LE3)

In [ ]:
steamcharts = add_rolling_mean(steamcharts, "avg_players", months=3)
df = add_steamcharts(df, steamcharts)

df = add_features(df, steamcharts)
df = add_groups(df)
df = add_scaled(df, "Hours_watched")
df = add_scaled(df, "avg_players")
df = flag_outliers(df, "Hours_watched")
df.shape

In [ ]:
# long -> wide: hours watched per game and month
make_wide_table(df).loc[["Grand Theft Auto V", "Counter-Strike: Global Offensive", "Dota 2"]].iloc[:, :6]

## First analysis

Twitch and Steam numbers are scaled per game (0 = worst month, 1 = best month), so we can see both curves in one plot.

In [ ]:
game = df[df["Game"] == "Among Us"].sort_values("date")

plt.figure(figsize=(10, 4))
plt.plot(game["date"], game["Hours_watched_scaled"], label="Twitch hours watched")
plt.plot(game["date"], game["avg_players_scaled"], label="Steam players")
plt.title("Among Us - Twitch vs Steam (scaled per game)")
plt.legend()
plt.show()

In [ ]:
# does the change on twitch go together with the change on steam?
#  - same month:  hours_watched_change vs players_change
#  - next month:  hours_watched_change vs players_next_change  (is twitch ahead of steam?)
df[["hours_watched_change", "players_change", "players_next_change"]].corr(method="spearman")

In [ ]:
# median players per twitch rank group and price group (ordered categories)
df.groupby("rank_group", observed=True)["avg_players"].median()